# Chapter 17 — Hypothesis Testing for Model Evaluation
*Track 3: Data Scientists*

## 🎯 Learning Objectives
- Apply statistical tests to compare ML model performance
- Understand paired tests vs. independent tests
- Avoid common pitfalls: multiple comparisons, data leakage in testing

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_val_score, KFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(42)
plt.style.use("seaborn-v0_8-whitegrid")
print("Libraries loaded ✅")

## 1. Concept Review — Testing Models, Not Just Parameters

When comparing two models, we ask:
*Is Model A's performance **significantly better** than Model B, or is the
difference just random fluctuation from the train/test split?*

Key tests:
- **Paired t-test** (same CV folds): corrected for fold correlation
- **McNemar's test**: for binary classifiers on same test set
- **Wilcoxon signed-rank**: non-parametric alternative

In [ ]:
# Load real dataset
data = load_breast_cancer()
X, y = data.data, data.target
scaler = StandardScaler()
X_sc = scaler.fit_transform(X)

kf = KFold(n_splits=10, shuffle=True, random_state=42)
lr_scores  = cross_val_score(LogisticRegression(max_iter=1000), X_sc, y, cv=kf, scoring="accuracy")
rf_scores  = cross_val_score(RandomForestClassifier(n_estimators=100, random_state=42), X_sc, y, cv=kf)

print(f"Logistic Regression: mean={lr_scores.mean():.4f}, std={lr_scores.std():.4f}")
print(f"Random Forest:       mean={rf_scores.mean():.4f}, std={rf_scores.std():.4f}")

# Paired t-test
t_stat, p_val = stats.ttest_rel(rf_scores, lr_scores)
print(f"
Paired t-test: t={t_stat:.3f}, p={p_val:.4f}")
print("RF is significantly better" if p_val < 0.05 else "No significant difference")

## 2. Math Walkthrough — Corrected Resampled t-Test

The standard paired t-test overestimates significance because CV folds
share training data. Nadeau & Bengio correction:

$$t = \frac{\bar d}{\sqrt{(\frac{1}{k} + \frac{n_{test}}{n_{train}}) \cdot \hat\sigma^2_d}}$$

where $d_i$ = performance difference on fold $i$.

In [ ]:
def corrected_ttest(scores_a, scores_b, n_train_ratio=0.9):
    diffs = scores_a - scores_b
    n = len(diffs)
    mean_diff = diffs.mean()
    var_diff = diffs.var(ddof=1)
    correction = 1/n + (1-n_train_ratio)/n_train_ratio
    t_stat = mean_diff / np.sqrt(correction * var_diff)
    p_val = 2 * stats.t.sf(abs(t_stat), df=n-1)
    return t_stat, p_val

t_corr, p_corr = corrected_ttest(rf_scores, lr_scores)
t_std, p_std = stats.ttest_rel(rf_scores, lr_scores)
print(f"Standard paired t:  t={t_std:.3f},  p={p_std:.4f}")
print(f"Corrected t-test:   t={t_corr:.3f}, p={p_corr:.4f}")

## 3–6. Multiple Comparisons, McNemar's, and Practice

In [ ]:
# Multiple comparisons — Bonferroni correction
models = {
    "LR": LogisticRegression(max_iter=1000),
    "RF": RandomForestClassifier(n_estimators=50, random_state=42),
    "GB": GradientBoostingClassifier(n_estimators=50, random_state=42),
}
all_scores = {name: cross_val_score(m, X_sc, y, cv=kf) for name, m in models.items()}

comparisons = [("RF", "LR"), ("GB", "LR"), ("RF", "GB")]
print("Model Comparison Results (α=0.05):")
for a, b in comparisons:
    t, p = stats.ttest_rel(all_scores[a], all_scores[b])
    p_bonf = min(p * len(comparisons), 1.0)
    sig = "✅ Significant" if p_bonf < 0.05 else "❌ Not significant"
    print(f"  {a} vs {b}: p={p:.4f}, p_bonf={p_bonf:.4f} — {sig}")

In [ ]:
# Practice: McNemar's test on binary predictions
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_sc, y, test_size=0.3, random_state=42)
lr = LogisticRegression(max_iter=1000).fit(X_train, y_train)
rf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train, y_train)

pred_lr = lr.predict(X_test)
pred_rf = rf.predict(X_test)

# Build 2x2 contingency table
b = ((pred_lr == y_test) & (pred_rf != y_test)).sum()  # LR right, RF wrong
c = ((pred_lr != y_test) & (pred_rf == y_test)).sum()  # LR wrong, RF right
print(f"McNemar: b={b}, c={c}")
chi2_mn = (abs(b-c)-1)**2 / (b+c)
p_mcnemar = 1 - stats.chi2.cdf(chi2_mn, df=1)
print(f"χ²={chi2_mn:.3f}, p={p_mcnemar:.4f}")
print("Significant difference" if p_mcnemar < 0.05 else "No significant difference")